# 3D-CNN: Rowing Stroke Quality Classifier

## Table of Contents
1. [Motivation](#motivation)
2. [Installation](#installation)
3. [Daten laden](#daten-laden)
4. [Bilder laden](#bilder-laden)
5. [Train/Test Split](#traintest-split)
6. [Modell-Architektur](#modell-architektur)
7. [Training](#training)
8. [Evaluation](#evaluation)
9. [Vergleich](#vergleich)
10. [Diskussion](#diskussion)


## Motivation

### Was macht ein 3D-CNN anders als ein 1D-CNN?

Das **1D-CNN** bekommt Gelenkwinkel als Zahlen — es "sieht" den Ruderer nie direkt.

Das **3D-CNN** bekommt die echten **Skeleton-Bilder** als Eingabe — es lernt direkt aus dem visuellen Erscheinungsbild des Schlags:

```
1D-CNN Eingabe:   [90°, 91°, 88°, ...] (Zahlen)

3D-CNN Eingabe:   Frame 1:  🖼️  (64×64 Pixelbild des Skeletts)
                  Frame 2:  🖼️
                  Frame 3:  🖼️
                     ...
                  Frame 28: 🖼️
```

### Wie ein 3D-Filter funktioniert

Ein 2D-Filter (für Bilder) schaut auf einen **Bereich im Raum** (z.B. 3×3 Pixel).
Ein 3D-Filter schaut auf einen **Bereich im Raum UND in der Zeit** gleichzeitig (z.B. 3×3 Pixel × 3 Frames):

```
Zeit →   Frame1    Frame2    Frame3
         ┌─────┐   ┌─────┐   ┌─────┐
         │░░░░░│   │░░░░░│   │░░░░░│
         │░[▓▓]│   │░[▓▓]│   │░[▓▓]│  ← 3D-Filter erfasst alle drei Frames gleichzeitig
         │░░░░░│   │░░░░░│   │░░░░░│
         └─────┘   └─────┘   └─────┘
```

Der Filter lernt: *"Wenn sich an dieser Körperstelle über diese drei Frames diese Bewegung zeigt, ist das ein Muster für einen guten/schlechten Schlag."*

### Warum Skeleton-Bilder statt Rohvideos?

Die Skeleton-Frames zeigen nur das Strichfigur-Skelett auf schwarzem Hintergrund — kein Wasser, kein Hintergrund, keine Kleidung. Das Modell lernt dadurch Körperbewegung, nicht Videoeigenschaften.


## Installation


In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec("tensorflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "-q"])
    print("TensorFlow installiert.")
else:
    import tensorflow as tf
    print(f"TensorFlow {tf.__version__} bereits vorhanden.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "requirements.txt").exists()
)
SKELETON_BASE = PROJECT_ROOT / "1_DatasetCharacteristics" / "Data" / "skeleton_frames"

print(f"TensorFlow {tf.__version__}")
print(f"Skeleton-Ordner: {SKELETON_BASE}")


## Daten laden

Die Skeleton-Bilder liegen mit **30 fps** vor. Um konsistent mit den anderen Notebooks zu bleiben (die die Landmarks bei 10 fps verwenden), laden wir jeden **3. Frame** — das entspricht 10 fps.

```
30-fps-Frames:   0   1   2   3   4   5   6   7   8   9  ...
10-fps-Auswahl:  ✓           ✓           ✓           ✓   ...
Skeleton-Datei: frame_00000  frame_00003  frame_00006  ...
```


In [ ]:
# Video-Informationen: (Name, Skeleton-Ordner, Anzahl 10fps-Frames, Anzahl Schläge, Label)
VIDEO_INFO = [
    ("cla-BAD",       "cla-BAD-skeleton",       506,  21, 0),  # 0 = BAD
    ("cla-GOOD-fast", "cla-GOOD-fast-skeleton", 2094, 88, 1),  # 1 = GOOD
    ("cla-GOOD-slow", "cla-GOOD-slow-skeleton", 1900, 68, 1),
    ("mar-GOOD-fast", "mar-GOOD-fast-skeleton", 1406, 62, 1),
    ("mar-GOOD-slow", "mar-GOOD-slow-skeleton", 1359, 51, 1),
]

# Schläge aufbauen: stroke_id → {folder, frame_indices (10fps), label}
stroke_info = {}  # stroke_id → {"folder": ..., "fps10_indices": [...], "label": ...}
global_sid = 0

for _, folder, n_frames_10fps, n_strokes, label in VIDEO_INFO:
    fps_per_stroke = n_frames_10fps / n_strokes
    for i in range(n_frames_10fps):
        sid = global_sid + int(i // fps_per_stroke)
        if sid not in stroke_info:
            stroke_info[sid] = {"folder": folder, "fps10_indices": [], "label": label}
        stroke_info[sid]["fps10_indices"].append(i)
    global_sid += n_strokes

stroke_lengths = [len(v["fps10_indices"]) for v in stroke_info.values()]
MAX_LEN = max(stroke_lengths) + 2

print(f"Schläge gesamt : {len(stroke_info)}")
print(f"Frames/Schlag  : Min={min(stroke_lengths)}  Max={max(stroke_lengths)}  Median={np.median(stroke_lengths):.0f}")
print(f"MAX_LEN        : {MAX_LEN}")

# Beispiel: Inhalt von Schlag 0
example = stroke_info[0]
print(f"\nSchlag 0: Label={example['label']} ({len(example['fps10_indices'])} Frames)")
print(f"  10fps-Frame 0 → Skeleton-Datei: frame_{example['fps10_indices'][0]*3:05d}.png")
print(f"  10fps-Frame 1 → Skeleton-Datei: frame_{example['fps10_indices'][1]*3:05d}.png")


## Bilder laden

Die Skeleton-Bilder sind **256×256 RGB**. Wir verkleinern sie auf **64×64 Graustufen** um:
- Den Speicherbedarf zu reduzieren
- Die Parameteranzahl des Modells klein zu halten (wichtig bei nur 290 Trainingsbeispielen)
- Farbe ist irrelevant — es geht um die Form des Skeletts

Das Laden dauert ein paar Minuten (7 265 Bilder).


In [ ]:
IMG_H, IMG_W = 64, 64

def load_skeleton_frame(folder, fps10_idx):
    """Lädt einen Skeleton-Frame: 10fps-Index → 30fps-Datei (stride 3)."""
    path = SKELETON_BASE / folder / f"frame_{fps10_idx * 3:05d}.png"
    img  = Image.open(path).convert("L")             # Graustufen
    img  = img.resize((IMG_W, IMG_H), Image.BILINEAR) # auf 64×64 verkleinern
    return np.array(img, dtype=np.float32) / 255.0    # Pixelwerte 0–1 normieren

# Beispielbild anzeigen
test_img = load_skeleton_frame("cla-BAD-skeleton", 0)
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(test_img, cmap="gray")
ax.set_title("Skeleton-Frame (64×64, Graustufen)")
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"Bildgröße: {test_img.shape}  Wertebereich: [{test_img.min():.2f}, {test_img.max():.2f}]")


In [ ]:
# Alle Schläge als 3D-Sequenzen laden
# Form des Arrays: (n_strokes, MAX_LEN, IMG_H, IMG_W, 1)
n_strokes = len(stroke_info)
X_3d = np.zeros((n_strokes, MAX_LEN, IMG_H, IMG_W, 1), dtype=np.float32)
y_3d = np.zeros(n_strokes, dtype=np.float32)

print(f"Lade {n_strokes} Schläge × bis zu {MAX_LEN} Frames × {IMG_H}×{IMG_W} Pixel ...")
print(f"Array-Speicher: {X_3d.nbytes / 1e6:.0f} MB")

t0 = time.time()
for i, sid in enumerate(sorted(stroke_info.keys())):
    info   = stroke_info[sid]
    folder = info["folder"]
    indices = info["fps10_indices"]
    n = min(len(indices), MAX_LEN)
    for j, fps10_idx in enumerate(indices[:n]):
        X_3d[i, j, :, :, 0] = load_skeleton_frame(folder, fps10_idx)
    y_3d[i] = info["label"]
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{n_strokes} Schläge geladen  ({elapsed:.1f}s)")

print(f"\nFertig in {time.time()-t0:.1f}s")
print(f"X_3d Shape : {X_3d.shape}")
print(f"           : ({n_strokes} Schläge × {MAX_LEN} Frames × {IMG_H} × {IMG_W} × 1 Kanal)")
print(f"GOOD-Schläge: {int(y_3d.sum())}  BAD-Schläge: {int((1-y_3d).sum())}")


In [ ]:
# Ersten Schlag visualisieren: 6 Frames aus dem Schlag nebeneinander
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, (sid, title) in enumerate([(0, "BAD-Schlag (sid=0)"), (21, "GOOD-Schlag (sid=21)")]):
    frame_indices = np.linspace(0, MAX_LEN - 1, 6, dtype=int)
    for ax, fi in zip(axes[row], frame_indices):
        ax.imshow(X_3d[sid, fi, :, :, 0], cmap="gray")
        ax.set_title(f"Frame {fi}", fontsize=8)
        ax.axis("off")
    axes[row, 0].set_ylabel(title, fontsize=9)
plt.suptitle("Skeleton-Sequenz: 6 Frames aus BAD- und GOOD-Schlag", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## Train/Test Split


In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(sss.split(X_3d, y_3d))

X_train, X_test = X_3d[train_idx], X_3d[test_idx]
y_train, y_test = y_3d[train_idx], y_3d[test_idx]

print(f"Training : {len(X_train)} Schläge  (GOOD: {int(y_train.sum())}  BAD: {int((1-y_train).sum())})")
print(f"Test     : {len(X_test)} Schläge  (GOOD: {int(y_test.sum())}  BAD: {int((1-y_test).sum())})")

# Klassengewichte gegen die 93/7-Ungleichverteilung
cw = compute_class_weight("balanced", classes=np.array([0.0, 1.0]), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}
print(f"\nKlassengewicht BAD: {cw[0]:.2f}  GOOD: {cw[1]:.2f}")


## Modell-Architektur

Das 3D-CNN verarbeitet Eingaben der Form **(Frames × Höhe × Breite × Kanäle)**.

```
Eingabe: (MAX_LEN × 64 × 64 × 1)
         ↓
┌──────────────────────────────────────────────────┐
│  Conv3D(16 Filter, Kern 3×3×3, relu)             │  ← erkennt räumlich-zeitliche Muster
│  BatchNormalization                              │
│  MaxPool3D(2×2×2)  → halbiert alle 3 Dimensionen│
└──────────────────────────────────────────────────┘
         ↓
┌──────────────────────────────────────────────────┐
│  Conv3D(32 Filter, Kern 3×3×3, relu)             │  ← kombiniert einfache Muster
│  BatchNormalization                              │
│  MaxPool3D(2×2×2)                               │
└──────────────────────────────────────────────────┘
         ↓
┌──────────────────────────────────────────────────┐
│  GlobalAveragePooling3D                          │  ← fasst alles zu einem Vektor zusammen
└──────────────────────────────────────────────────┘
         ↓
┌──────────────────────────────────────────────────┐
│  Dense(32, relu)  +  Dropout(0.5)                │
└──────────────────────────────────────────────────┘
         ↓
┌──────────────────────────────────────────────────┐
│  Dense(1, sigmoid)  → Wahrscheinlichkeit GOOD    │
└──────────────────────────────────────────────────┘
```

Kleine Filter-Anzahl (16/32) weil wir nur 290 Trainingsbeispiele haben — zu viele Parameter würden zum Auswendiglernen führen.


In [ ]:
def build_3d_cnn(max_len, img_h, img_w):
    inputs = keras.Input(shape=(max_len, img_h, img_w, 1), name="stroke_video")

    x = layers.Conv3D(16, kernel_size=(3, 3, 3), padding="same", activation="relu", name="conv3d_1")(inputs)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPool3D(pool_size=(2, 2, 2), name="pool1")(x)

    x = layers.Conv3D(32, kernel_size=(3, 3, 3), padding="same", activation="relu", name="conv3d_2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPool3D(pool_size=(2, 2, 2), name="pool2")(x)

    x = layers.GlobalAveragePooling3D(name="global_pool")(x)

    x = layers.Dense(32, activation="relu", name="dense")(x)
    x = layers.Dropout(0.5, name="dropout")(x)

    output = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inputs, output, name="3D_CNN_Stroke_Classifier")


model = build_3d_cnn(MAX_LEN, IMG_H, IMG_W)
model.summary()


## Training


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=8,           # kleiner Batch wegen weniger Daten und großem Array
    validation_split=0.15,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(
    axes,
    ["loss", "accuracy"],
    ["Loss", "Accuracy"]
):
    ax.plot(history.history[metric],          label="Training",    color="steelblue")
    ax.plot(history.history[f"val_{metric}"], label="Validierung", color="darkorange")
    ax.set_xlabel("Epoche")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle("Trainingsverlauf 3D-CNN", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Training gestoppt nach Epoche {len(history.history['loss'])}.")


## Evaluation


In [ ]:
y_proba = model.predict(X_test, verbose=0).flatten()
y_pred  = (y_proba >= 0.5).astype(int)

auc    = roc_auc_score(y_test, y_proba)
f1_mac = f1_score(y_test, y_pred, average="macro", zero_division=0)
report = classification_report(y_test, y_pred, target_names=["BAD", "GOOD"], output_dict=True, zero_division=0)

print(f"{'='*54}")
print(f"  3D-CNN Evaluation (Test Set)")
print(f"{'='*54}")
print(f"  ROC-AUC     : {auc:.4f}")
print(f"  Macro F1    : {f1_mac:.4f}")
print(f"  BAD Recall  : {report['BAD']['recall']:.4f}  ← % der BAD-Schläge erkannt")
print(f"  BAD Prec.   : {report['BAD']['precision']:.4f}  ← % der Alarme wirklich BAD")
print()
print(classification_report(y_test, y_pred, target_names=["BAD", "GOOD"], zero_division=0))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["BAD", "GOOD"]
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — 3D-CNN")

RocCurveDisplay.from_predictions(y_test, y_proba, name="3D-CNN", ax=axes[1], color="steelblue")
axes[1].plot([0, 1], [0, 1], "k--", label="Zufall")
axes[1].set_title("ROC-Kurve — 3D-CNN")
axes[1].legend()

plt.suptitle("3D-CNN Ergebnisse (Test Set)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## Vergleich


In [ ]:
# Isolation Forest auf Schlag-Ebene (Gelenkwinkel-Aggregation) zum Vergleich
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Gelenkwinkel neu berechnen (kompakt, gleiche Logik wie andere Notebooks)
DATA_PATH = PROJECT_ROOT / "1_DatasetCharacteristics" / "landmarks" / "landmarks_all_10fps.csv"
df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c not in ("Good Stroke", "Bad Stroke")]
X_raw   = df[feature_cols].values
y_frame = df["Good Stroke"].values

stroke_ids = []
global_sid = 0
for _, folder, n_frames, n_strokes, label in VIDEO_INFO:
    for i in range(n_frames):
        stroke_ids.append(global_sid + int(i // (n_frames / n_strokes)))
    global_sid += n_strokes
stroke_ids = np.array(stroke_ids)

ANGLE_DEFS = [
    ("left_wrist",  "left_elbow",    "left_shoulder"),
    ("right_wrist", "right_elbow",   "right_shoulder"),
    ("left_elbow",  "left_shoulder", "left_hip"),
    ("right_elbow", "right_shoulder","right_hip"),
    ("left_shoulder","left_hip",     "left_knee"),
    ("right_shoulder","right_hip",   "right_knee"),
    ("left_hip",    "left_knee",     "left_ankle"),
    ("right_hip",   "right_knee",    "right_ankle"),
]

df_lm = pd.DataFrame(X_raw, columns=feature_cols)
def get_xyz(lm): return df_lm[[f"{lm}_x",f"{lm}_y",f"{lm}_z"]].values
def angle(a,b,c):
    ba=get_xyz(a)-get_xyz(b); bc=get_xyz(c)-get_xyz(b)
    d=np.linalg.norm(ba,axis=1)*np.linalg.norm(bc,axis=1)+1e-9
    return np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i",ba,bc)/d,-1,1)))

ang_mat = np.column_stack([angle(a,b,c) for a,b,c in ANGLE_DEFS])
df_ang  = pd.DataFrame(ang_mat, columns=[f"a{i}" for i in range(8)])
df_ang["sid"] = stroke_ids
ANG_COLS = [f"a{i}" for i in range(8)]

m = df_ang.groupby("sid")[ANG_COLS].mean()
s = df_ang.groupby("sid")[ANG_COLS].std().fillna(0)
X_flat = np.hstack([m.values, s.values])
y_flat = np.array([y_frame[stroke_ids == sid][0] for sid in np.unique(stroke_ids)])

# Gleichen Split wie 3D-CNN verwenden
X_flat_train, X_flat_test = X_flat[train_idx], X_flat[test_idx]
y_flat_train, y_flat_test = y_flat[train_idx], y_flat[test_idx]

scaler = StandardScaler()
Xf_tr = scaler.fit_transform(X_flat_train)
Xf_te = scaler.transform(X_flat_test)

# One-class (nur GOOD im Training)
iso = IsolationForest(n_estimators=200, contamination=0.1, random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(Xf_tr[y_flat_train == 1])
iso_scores = iso.score_samples(Xf_te)
iso_pred   = (iso.predict(Xf_te) == 1).astype(int)

print(f"Test-Set: {len(y_flat_test)} Schläge — GOOD: {int(y_flat_test.sum())}  BAD: {int((1-y_flat_test).sum())}")


In [ ]:
def get_metrics(y_true, y_proba_or_score, threshold=0.5):
    pred   = (y_proba_or_score >= threshold).astype(int)
    auc    = roc_auc_score(y_true, y_proba_or_score)
    f1m    = f1_score(y_true, pred, average="macro", zero_division=0)
    rep    = classification_report(y_true, pred, target_names=["BAD","GOOD"], output_dict=True, zero_division=0)
    return auc, f1m, rep["BAD"]["recall"]

cnn3_auc, cnn3_f1, cnn3_rec = get_metrics(y_test,      y_proba)
iso_auc,  iso_f1,  iso_rec  = get_metrics(y_flat_test, iso_scores,
                                           threshold=np.percentile(iso_scores, 10))

print(f"{'Modell':<30} {'ROC-AUC':>8} {'Macro F1':>9} {'BAD Recall':>11} {'Eingabe':>18}")
print("-" * 81)
print(f"  {'3D-CNN':<28} {cnn3_auc:>8.4f} {cnn3_f1:>9.4f} {cnn3_rec:>11.4f} {'Skeleton-Bilder':>18}")
print(f"  {'Isolation Forest':<28} {iso_auc:>8.4f} {iso_f1:>9.4f} {iso_rec:>11.4f} {'Gelenkwinkel':>18}")
print()
print("Beide Modelle verwenden denselben Test-Split.")


## Diskussion

### Was das 3D-CNN anders macht

Das 3D-CNN ist das einzige Modell in diesem Projekt, das **nie manuell berechnete Features** benutzt. Es sieht nur die rohen Skeleton-Bilder und muss selbst lernen, welche Pixelmuster auf gute oder schlechte Technik hindeuten.

Das ist konzeptuell elegant — aber es hat seinen Preis.

### Warum es bei diesem Datensatz schwierig ist

| Problem | Auswirkung |
|---------|------------|
| Nur 290 Schläge | Das 3D-CNN hat viele Parameter — es lernt schnell auswendig statt zu generalisieren |
| 21 BAD aus einem Video | Modell kann optische Videomerkmale lernen, nicht Technik |
| Skeleton-Bilder sind spärlich | Strichfigur auf schwarzem Hintergrund — wenig Pixelinformation |

### Wann ein 3D-CNN sinnvoll wäre

- **Viele Daten**: Mehrere tausend Schläge von vielen verschiedenen Ruderern
- **Verschiedene BAD-Videos**: BAD-Technik von mehreren Personen mit unterschiedlichem Aussehen
- **Rohvideos**: Wenn keine Landmark-Extraktion möglich ist und man direkt auf dem Video arbeiten muss

### Vergleich der Ansätze

| Ansatz | Eingabe | Vorteil | Nachteil |
|--------|---------|---------|----------|
| One-Class (IF/SVM) | Winkel-Statistiken | Kein BAD nötig | Verliert Zeitinfo |
| 1D-CNN | Winkelsequenz | Nutzt Zeitstruktur | Kleiner Datensatz |
| **3D-CNN** | **Skeleton-Bilder** | **Kein Feature Engineering** | **Braucht viele Daten** |

### Fazit

Das 3D-CNN ist das mächtigste und flexibelste Modell — aber auch das datenintensivste. Für dieses Projekt mit 290 Schlägen ist es mit hoher Wahrscheinlichkeit überparametrisiert. Die Metriken sollten kritisch bewertet werden: ein hoher Testwert kann Auswendiglernen bedeuten, da Training und Test beide Schläge aus `cla-BAD` enthalten.
